# 🧠 Entrenamiento de un Traductor Neuronal Shiwilu–Español
**Objetivo:** Entrenar un modelo de traducción automática neuronal (NMT) desde cero con corpus paralelo Shiwilu ↔ Español.
Se usarán tecnologías modernas como *Hugging Face Transformers*, *datasets*, *sentencepiece* y *fairseq/mBART*.

🔬 Este notebook asume que el corpus está alineado en dos archivos: `shiwilu.txt` y `espanol.txt`, una oración por línea.


## 1️⃣ Preparación del entorno

In [ ]:
# Instalar librerías necesarias
!pip install -q transformers datasets sentencepiece sacrebleu

## 2️⃣ Carga y preprocesamiento del corpus

In [ ]:
from datasets import Dataset

# Cargar corpus paralelo (suponiendo archivos en mismo directorio)
with open("shiwilu.txt", encoding='utf-8') as f:
    shiwilu = [line.strip() for line in f.readlines()]

with open("espanol.txt", encoding='utf-8') as f:
    espanol = [line.strip() for line in f.readlines()]

data = Dataset.from_dict({"translation": [{"shw": s, "es": e} for s, e in zip(shiwilu, espanol)]})
data = data.train_test_split(test_size=0.1)
data

## 3️⃣ Tokenización y codificación (con SentencePiece BPE)

In [ ]:
from transformers import MarianTokenizer, MarianMTModel, Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import DatasetDict

# Usar tokenizer multilingüe como mBART o entrenar uno propio con sentencepiece (a futuro)
# Aquí ejemplo con mBART
model_name = 'facebook/mbart-large-50-many-to-many-mmt'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

SRC_LANG = "shw_XX"  # código artificial o real si se entrena uno
TGT_LANG = "es_XX"

def preprocess_function(example):
    inputs = tokenizer(example["translation"]["shw"], truncation=True, padding="max_length", max_length=64, return_tensors="pt")
    targets = tokenizer(example["translation"]["es"], truncation=True, padding="max_length", max_length=64, return_tensors="pt")
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_data = data.map(preprocess_function, batched=True, remove_columns=["translation"])
tokenized_data

## 4️⃣ Configurar entrenamiento

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./nmt_shiwilu",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    save_total_limit=2,
    predict_with_generate=True,
    fp16=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["test"]
)
# Entrenamiento (puede tardar bastante tiempo)
# trainer.train()

## 5️⃣ Evaluación del modelo

In [ ]:
from transformers import pipeline

translator = pipeline("translation", model=model, tokenizer=tokenizer)
print(translator("ashintu niwan", src_lang=SRC_LANG, tgt_lang=TGT_LANG))

## 🔚 Conclusiones y extensiones
- Considera recolectar más corpus paralelo.
- Entrena un tokenizer específico con SentencePiece para Shiwilu.
- Evalúa con BLEU, chrF y humana.
- Intenta transfer learning o fine-tuning sobre mBART o NLLB.
